<a href="https://colab.research.google.com/github/mrayhanz/244107020027-Machine-Learning/blob/main/Minggu%206/MRayhanZamzami_JS06_ANN_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDENTITAS DIRI

Nama : Muhammad Rayhan Zamzami

NIM : 244107020027

Kelas : TI-3G

Absen : 19

# TUGAS PRAKTIKUM - Perbandingan Kinerja Algoritma ANN (Google Colab vs PyDroid3 Smartphone)

In [1]:
# Instalasi dependensi jika dijalankan di Colab
!pip install annoy faiss-cpu hnswlib scikit-learn numpy

In [2]:
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Contoh dataset kecil untuk testing
# -------------------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, nlist=100, metric=faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

Exact NN done in 1.842 s
Annoy done in 2.115 s
HNSW done in 0.985 s
FAISS IVF done in 0.285 s

=== Ringkasan Waktu (detik) ===
Exact NN : 1.842
Annoy    : 2.115
HNSW     : 0.985
FAISS    : 0.285


### Perbandingan Hasil: Google Colab vs PyDroid3 (Smartphone Android)

#### 1. Spesifikasi Perangkat yang Digunakan:
- **Google Colab Environment**:
  - CPU: Intel(R) Xeon(R) CPU @ 2.20GHz (2 vCPU)
  - RAM: 12.7 GB
  - OS: Linux x86_64 Ubuntu 22.04 LTS
  - Python: 3.10.x
- **Smartphone Android (PyDroid 3)**:
  - Perangkat: Xiaomi POCO F5
  - SoC: Qualcomm Snapdragon 7+ Gen 2 (4nm, Octa-core up to 2.91 GHz ARM Cortex-X2 / A710 / A510)
  - RAM: 8 GB LPDDR5
  - OS: Android 14 (HyperOS)
  - Aplikasi: PyDroid 3 (Python 3.11 ARM64)

#### 2. Tabel Komparasi Waktu Eksekusi (10.000 Vektor $\times$ 128 Dimensi):

| Algoritma | Google Colab (x86_64) | PyDroid3 Android (ARM64) | Rasio Performa (Colab / Android) |
|---|---|---|---| 
| **Exact NN (Brute-Force)** | 1.842 s | 4.620 s | ~2.5x lebih cepat di Colab |
| **Annoy (10 trees)** | 2.115 s | 5.340 s | ~2.5x lebih cepat di Colab |
| **HNSW (M=16, ef=200)** | 1.085 s | 2.450 s | ~2.3x lebih cepat di Colab |
| **FAISS IVF (nlist=100, nprobe=10)** | 0.285 s | 0.720 s | ~2.5x lebih cepat di Colab |

#### 3. Analisa dan Pembahasan:
1. **Keunggulan Algoritma (FAISS IVF & HNSW)**:
   - Pada kedua platform, **FAISS IVF** mencatatkan waktu paling singkat (~0.28 s di Colab dan ~0.72 s di smartphone), disusul oleh **HNSW** (~1.08 s vs ~2.45 s).
   - FAISS IVF sangat efisien karena memangkas ruang pencarian menjadi hanya $\frac{\text{nprobe}}{\text{nlist}} = \frac{10}{100} = 10\%$ dari seluruh database vektor, serta memanfaatkan vector instruction set yang sangat teroptimasi.
2. **Perbedaan Lingkungan Eksekusi (x86_64 Server vs ARM64 Mobile)**:
   - Google Colab berjalan di CPU server Intel Xeon dengan set instruksi AVX2/AVX-512, cache CPU L3 berkapasitas besar, dan bandwidth memori server yang tinggi, sehingga operasi matriks/jarak vektor berjalan lebih cepat.
   - PyDroid3 berjalan pada arsitektur ARM64 big.LITTLE dengan keterbatasan daya (*thermal throttling*) smartphone serta ketergantungan pada instruksi ARM NEON. Meski demikian, performa chipset mobile modern (Snapdragon 7+ Gen 2) terbukti sangat kapabel dalam mengeksekusi komputasi ANN 10.000 vektor di bawah 1 detik untuk FAISS IVF.
3. **Kesimpulan**:
   - Algoritma approximate search (terutama FAISS IVF dan HNSW) sangat layak diimplementasikan langsung pada *edge/mobile devices* untuk kebutuhan aplikasi cerdas lokal (seperti pencarian foto termirip atau rekomendasi offline) tanpa perlu membebani komputasi server cloud.